# GeoAtt-PointNet++ — Training (Local)
Rahmat Zulfikri · Magister Teknik Elektro UGM

Notebook ini untuk training **di komputer lokal** (bukan Google Colab).  
Menggunakan modul proyek yang sudah ada di `models/`, `utils/`, `losses/`, dan `train.py`.

**Fitur:**
- 🛑 Early Stopping — berhenti otomatis jika val_loss tidak membaik
- 🔧 Two-Phase Training — Phase 1 (main) + Phase 2 (fine-tune LR kecil)
- ⚖️ Dataset Balancing — cap setiap label ke jumlah sesi minimum
- 📊 Live loss plot setelah training selesai

---
### Sections:
1. Setup & Cek Environment
2. Konfigurasi
3. Dataset — Scan, Balance, Split
4. Training (LOSO atau Single Fold)
5. Plot Loss Curves

## 1. Setup & Cek Environment

In [ ]:
# ── Import standard ───────────────────────────────────────────────────────────
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Tambahkan root proyek ke path agar bisa import modul lokal
PROJECT_ROOT = Path().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ── Import modul proyek ───────────────────────────────────────────────────────
import torch
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR, StepLR
from torch.utils.data import DataLoader

from losses.contrastive import ContrastiveLoss
from models.siamese import SiamesePalmNet
from train import EarlyStopping, _run_epoch, _save_checkpoint
from utils.augmentation import PointCloudAugmentor, GeometryAugmentor
from utils.dataset import (
    PalmPairDataset,
    balance_label_frames,
    balance_label_sessions,
    load_geometry,
    make_loso_splits,
    make_loso_splits_frames,
    scan_dataset,
    scan_dataset_frames,
)
from utils.normalizer import GeometryNormalizer

warnings.filterwarnings('ignore')

# ── Cek device ────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else
                      'mps'  if torch.backends.mps.is_available() else
                      'cpu')
print(f'PyTorch : {torch.__version__}')
print(f'Device  : {device}')
if device.type == 'cuda':
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
elif device.type == 'mps':
    print('GPU     : Apple Silicon MPS')
else:
    print('GPU     : tidak ada (CPU mode — training akan lebih lambat)')
print(f'Project : {PROJECT_ROOT}')

## 2. Konfigurasi

Ubah parameter di sini sesuai kebutuhan. Semua path bersifat **relatif** terhadap folder proyek.

In [ ]:
# ── Path ──────────────────────────────────────────────────────────────────────
DATA_DIR   = PROJECT_ROOT / 'dataset'
OUTPUT_DIR = PROJECT_ROOT / 'runs' / 'local_exp1'

# ── Dataset ───────────────────────────────────────────────────────────────────
BALANCE_DATASET = True
SEED            = 42

# ── Model ─────────────────────────────────────────────────────────────────────
# 14 fitur geometri: finger_lengths×5, palm_width, palm_height, palm_depth_std,
#                    finger_widths×5, mean_palm_curvature
GEOM_DIM  = 14
N_POINTS  = 4096
SAMPLING  = 'random'  # 'random' atau 'fps'

# ── Training Phase 1 (Main) ───────────────────────────────────────────────────
EPOCHS      = 100
BATCH_SIZE  = 8      # kurangi jika RAM terbatas (16 untuk GPU, 8 untuk CPU)
LR          = 1e-3
MARGIN      = 0.5
PATIENCE    = 15     # early stopping: berhenti jika tidak membaik selama N epoch
MIN_DELTA   = 1e-4
NUM_WORKERS = 0      # 0 = aman untuk macOS / Windows lokal

# ── Training Phase 2 (Fine-Tune) ──────────────────────────────────────────────
FINETUNE_EPOCHS = 20
FINETUNE_LR     = 1e-4

# ── Mode training ─────────────────────────────────────────────────────────────
# 'loso'  = Leave-One-Session-Out cross validation (semua fold)
# 'single'= train satu fold saja (lebih cepat untuk eksplorasi)
# 'quick' = 1 fold + epoch dikurangi (untuk cek pipeline)
TRAIN_MODE = 'single'
FOLD_IDX   = 0

# ── Resume ────────────────────────────────────────────────────────────────────
RESUME_FROM = None

print('Konfigurasi:')
print(f'  DATA_DIR        : {DATA_DIR}')
print(f'  OUTPUT_DIR      : {OUTPUT_DIR}')
print(f'  BALANCE_DATASET : {BALANCE_DATASET}')
print(f'  GEOM_DIM        : {GEOM_DIM}')
print(f'  N_POINTS        : {N_POINTS}')
print(f'  BATCH_SIZE      : {BATCH_SIZE}')
print(f'  EPOCHS          : {EPOCHS}  (patience={PATIENCE})')
print(f'  FINETUNE_EPOCHS : {FINETUNE_EPOCHS}  (lr={FINETUNE_LR})')
print(f'  TRAIN_MODE      : {TRAIN_MODE}')
if not DATA_DIR.exists():
    print(f'  ⚠️  DATA_DIR tidak ditemukan: {DATA_DIR}')
    print("     Ubah DATA_DIR ke path yang benar.")

## 3. Dataset — Scan, Balance, Split

In [ ]:
# ── Auto-detect layout ────────────────────────────────────────────────────────
def _is_frame_layout(data_dir):
    for label_dir in data_dir.iterdir():
        if not label_dir.is_dir(): continue
        for ts_dir in label_dir.iterdir():
            if not ts_dir.is_dir(): continue
            if any(ts_dir.glob('frame_*')): return True
    return False

assert DATA_DIR.exists(), f"DATA_DIR tidak ditemukan: {DATA_DIR}"

frame_layout = _is_frame_layout(DATA_DIR)
print(f'Layout terdeteksi: {"frame (single-frame)" if frame_layout else "session (ICP)"}')

# ── Scan + balance ────────────────────────────────────────────────────────────
if frame_layout:
    label_frames, session_groups = scan_dataset_frames(DATA_DIR)
    if BALANCE_DATASET:
        session_groups, min_s = balance_label_frames(session_groups, seed=SEED)
        label_frames = {
            label: [f for ts_frames in ts_dict.values() for f in ts_frames]
            for label, ts_dict in session_groups.items()
        }
        print(f'[Balance] Setiap label = {min_s} sesi')
    n_labels   = len(label_frames)
    n_sessions = sum(len(ts) for ts in session_groups.values())
    n_frames   = sum(len(v) for v in label_frames.values())
    loso_splits = list(make_loso_splits_frames(session_groups))
    label_for_display = label_frames
else:
    label_sessions = scan_dataset(DATA_DIR)
    if BALANCE_DATASET:
        label_sessions, min_s = balance_label_sessions(label_sessions, seed=SEED)
        print(f'[Balance] Setiap label = {min_s} sesi')
    n_labels   = len(label_sessions)
    n_sessions = sum(len(v) for v in label_sessions.values())
    n_frames   = n_sessions
    loso_splits = list(make_loso_splits(label_sessions))
    label_for_display = label_sessions

print(f'\nLabel   : {n_labels}')
print(f'Sesi    : {n_sessions}')
if frame_layout:
    print(f'Frame   : {n_frames}')
print(f'LOSO folds: {len(loso_splits)}')
print(f'\nDistribusi per label:')

if frame_layout:
    for label in sorted(label_frames):
        ns = len(session_groups.get(label, {}))
        nf = len(label_frames[label])
        print(f'  {label:<18} {ns} sesi  {nf} frame')
else:
    for label, sessions in sorted(label_sessions.items()):
        print(f'  {label:<18} {len(sessions)} sesi')

assert n_labels >= 2, 'Butuh minimal 2 label untuk membentuk pasangan impostor'

## 4. Training

Training menggunakan two-phase approach:
- **Phase 1**: Adam + StepLR + Early Stopping
- **Phase 2**: Adam + CosineAnnealingLR + Early Stopping (LR lebih kecil)

Model terbaik (val_loss terendah) disimpan otomatis ke `best.pth`.

In [ ]:
def train_fold(fold_idx, train_sessions, test_sessions, output_dir):
    """
    Training satu LOSO fold dengan two-phase training + early stopping.
    Mengembalikan history loss untuk plotting.
    """
    output_dir = Path(output_dir) / f'fold_{fold_idx}'
    output_dir.mkdir(parents=True, exist_ok=True)

    # Fit normalizer pada training geometry saja (hindari data leakage)
    all_train_dirs = [s for ss in train_sessions.values() for s in ss]
    train_geoms    = [load_geometry(d) for d in all_train_dirs]
    normalizer     = GeometryNormalizer()
    normalizer.fit(train_geoms)
    normalizer.save(output_dir / 'normalizer.json')
    print(f'Normalizer fitted dari {len(train_geoms)} sesi training')

    # Augmentor: point cloud + geometry (hanya training)
    pc_augmentor   = PointCloudAugmentor()
    geom_augmentor = GeometryAugmentor(noise_sigma=0.02)

    train_ds = PalmPairDataset(
        label_sessions=train_sessions, n_points=N_POINTS,
        sampling=SAMPLING, augment=pc_augmentor,
        geom_augment=geom_augmentor, normalizer=normalizer)
    val_ds   = PalmPairDataset(
        label_sessions=test_sessions, n_points=N_POINTS,
        sampling=SAMPLING, augment=None,
        geom_augment=None, normalizer=normalizer)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=(device.type == 'cuda'), drop_last=True)
    val_loader   = DataLoader(
        val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    model     = SiamesePalmNet(geom_dim=GEOM_DIM).to(device)
    criterion = ContrastiveLoss(margin=MARGIN)
    history   = {'phase': [], 'epoch': [], 'train_loss': [], 'val_loss': [], 'lr': []}

    # ── Resume ────────────────────────────────────────────────────────────────
    start_epoch   = 0
    best_val_loss = float('inf')
    if RESUME_FROM and Path(RESUME_FROM).exists():
        ckpt = torch.load(RESUME_FROM, map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])
        start_epoch   = ckpt['epoch'] + 1
        best_val_loss = ckpt.get('best_val_loss', best_val_loss)
        print(f'Resume dari epoch {start_epoch}')

    # ── Phase 1: Main Training ────────────────────────────────────────────────
    print(f'\n--- Phase 1: Main Training (lr={LR:.2e}, max={EPOCHS} epoch, patience={PATIENCE}) ---')
    optimizer  = Adam(model.parameters(), lr=LR)
    scheduler  = StepLR(optimizer, step_size=30, gamma=0.5)
    early_stop = EarlyStopping(patience=PATIENCE, min_delta=MIN_DELTA)
    log_path   = output_dir / 'train_log.csv'

    with open(log_path, 'w') as f:
        f.write('phase,epoch,train_loss,val_loss,lr\n')

    for epoch in range(start_epoch, EPOCHS):
        t0         = time.time()
        train_loss = _run_epoch(model, train_loader, criterion, optimizer, device, train=True)
        val_loss   = _run_epoch(model, val_loader,   criterion, None,      device, train=False)
        scheduler.step()
        current_lr = scheduler.get_last_lr()[0]

        history['phase'].append(1)
        history['epoch'].append(epoch + 1)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['lr'].append(current_lr)

        print(f'P1 Epoch {epoch+1:03d}/{EPOCHS}  '
              f'train={train_loss:.4f}  val={val_loss:.4f}  '
              f'lr={current_lr:.2e}  t={time.time()-t0:.1f}s')

        with open(log_path, 'a') as f:
            f.write(f'1,{epoch+1},{train_loss:.6f},{val_loss:.6f},{current_lr:.6e}\n')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            _save_checkpoint(output_dir, model, optimizer, epoch, best_val_loss, fold_idx, 'best.pth')
            print(f'  ★ best.pth disimpan (val_loss={best_val_loss:.4f})')

        if early_stop.step(val_loss, model, epoch + 1):
            print(f'  [EarlyStopping] Berhenti di epoch {epoch+1}')
            break

    early_stop.restore_best(model, device)

    # ── Phase 2: Fine-Tuning ──────────────────────────────────────────────────
    if FINETUNE_EPOCHS > 0:
        print(f'\n--- Phase 2: Fine-Tuning '
              f'(lr={FINETUNE_LR:.2e}, max={FINETUNE_EPOCHS} epoch, patience={max(5, PATIENCE//2)}) ---')
        ft_optimizer = Adam(model.parameters(), lr=FINETUNE_LR)
        ft_scheduler = CosineAnnealingLR(ft_optimizer, T_max=FINETUNE_EPOCHS, eta_min=1e-6)
        ft_early     = EarlyStopping(patience=max(5, PATIENCE // 2), min_delta=MIN_DELTA)
        ft_best_loss = best_val_loss

        for ft_epoch in range(FINETUNE_EPOCHS):
            t0         = time.time()
            train_loss = _run_epoch(model, train_loader, criterion, ft_optimizer, device, train=True)
            val_loss   = _run_epoch(model, val_loader,   criterion, None,          device, train=False)
            ft_scheduler.step()
            current_lr = ft_scheduler.get_last_lr()[0]

            history['phase'].append(2)
            history['epoch'].append(ft_epoch + 1)
            history['train_loss'].append(train_loss)
            history['val_loss'].append(val_loss)
            history['lr'].append(current_lr)

            print(f'P2 Epoch {ft_epoch+1:03d}/{FINETUNE_EPOCHS}  '
                  f'train={train_loss:.4f}  val={val_loss:.4f}  '
                  f'lr={current_lr:.2e}  t={time.time()-t0:.1f}s')

            with open(log_path, 'a') as f:
                f.write(f'2,{ft_epoch+1},{train_loss:.6f},{val_loss:.6f},{current_lr:.6e}\n')

            if val_loss < ft_best_loss:
                ft_best_loss  = val_loss
                best_val_loss = ft_best_loss
                _save_checkpoint(output_dir, model, ft_optimizer, ft_epoch,
                                 best_val_loss, fold_idx, 'best.pth')
                print(f'  ★ best.pth (FT) disimpan (val_loss={best_val_loss:.4f})')

            if ft_early.step(val_loss, model, ft_epoch + 1):
                print(f'  [EarlyStopping] Fine-tune berhenti di epoch {ft_epoch+1}')
                break

        ft_early.restore_best(model, device)
        _save_checkpoint(output_dir, model, ft_optimizer,
                         FINETUNE_EPOCHS - 1, best_val_loss, fold_idx, 'best_finetuned.pth')

    print(f'\nFold {fold_idx} selesai. Best val_loss = {best_val_loss:.4f}')
    return history, best_val_loss

In [ ]:
# ── Jalankan training ─────────────────────────────────────────────────────────
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
all_histories = []
all_results   = []

if TRAIN_MODE == 'loso':
    folds_to_run = loso_splits
elif TRAIN_MODE == 'single':
    folds_to_run = [loso_splits[FOLD_IDX % len(loso_splits)]]
elif TRAIN_MODE == 'quick':
    EPOCHS = min(EPOCHS, 5)
    FINETUNE_EPOCHS = min(FINETUNE_EPOCHS, 3)
    folds_to_run = [loso_splits[0]]
    print(f'[Quick mode] EPOCHS={EPOCHS}, FINETUNE_EPOCHS={FINETUNE_EPOCHS}')
else:
    raise ValueError(f'TRAIN_MODE tidak valid: {TRAIN_MODE}')

t_total = time.time()
for fold_idx, train_sessions, test_sessions in folds_to_run:
    print(f'\n{"="*60}')
    print(f'Fold {fold_idx}  |  Train: {sum(len(v) for v in train_sessions.values())} item  '
          f'|  Test: {sum(len(v) for v in test_sessions.values())} item')
    print(f'{"="*60}')
    history, best_loss = train_fold(fold_idx, train_sessions, test_sessions, OUTPUT_DIR)
    all_histories.append((fold_idx, history))
    all_results.append({'fold': fold_idx, 'best_val_loss': best_loss})

elapsed = time.time() - t_total
print(f'\n{"="*60}')
print(f'Training selesai dalam {elapsed/60:.1f} menit')
print(f'{"─"*60}')
print(f'{"Fold":<8} {"Best Val Loss":>15}')
print(f'{"─"*60}')
for r in all_results:
    print(f'{r["fold"]:<8} {r["best_val_loss"]:>15.4f}')
print(f'{"─"*60}')
print(f'Output disimpan di: {OUTPUT_DIR}')

## 5. Plot Loss Curves

In [ ]:
def plot_training_history(histories, save_dir=None):
    """
    Plot loss curves untuk semua fold.
    Phase 1 (main) = solid line, Phase 2 (fine-tune) = dashed line.
    Garis vertikal memisahkan kedua fase.
    """
    n_folds = len(histories)
    if n_folds == 0:
        print('Tidak ada history untuk diplot.')
        return

    n_cols  = min(n_folds, 3)
    n_rows  = (n_folds + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(6 * n_cols, 4 * n_rows),
                             squeeze=False)

    for ax_idx, (fold_idx, h) in enumerate(histories):
        row, col = divmod(ax_idx, n_cols)
        ax = axes[row][col]

        phases = np.array(h['phase'])
        epochs = np.array(h['epoch'])
        tr     = np.array(h['train_loss'])
        vl     = np.array(h['val_loss'])

        # Hitung x-axis kumulatif (Phase 2 lanjut setelah Phase 1)
        p1_mask = phases == 1
        p2_mask = phases == 2
        p1_len  = p1_mask.sum()
        x_all   = np.concatenate([
            epochs[p1_mask],
            p1_len + epochs[p2_mask]
        ])
        tr_all = np.concatenate([tr[p1_mask], tr[p2_mask]])
        vl_all = np.concatenate([vl[p1_mask], vl[p2_mask]])

        ax.plot(x_all[:p1_len], tr_all[:p1_len],
                color='steelblue', lw=1.5, label='Train (P1)')
        ax.plot(x_all[:p1_len], vl_all[:p1_len],
                color='tomato', lw=1.5, label='Val (P1)')

        if p2_mask.any():
            ax.plot(x_all[p1_len:], tr_all[p1_len:],
                    color='steelblue', lw=1.5, linestyle='--', label='Train (P2)')
            ax.plot(x_all[p1_len:], vl_all[p1_len:],
                    color='tomato', lw=1.5, linestyle='--', label='Val (P2)')
            ax.axvline(p1_len, color='gray', lw=1, linestyle=':')
            ax.text(p1_len + 0.3, ax.get_ylim()[1] * 0.95,
                    'Fine-tune', fontsize=8, color='gray')

        best_val   = min(vl_all)
        best_epoch = x_all[np.argmin(vl_all)]
        ax.axvline(best_epoch, color='green', lw=1, linestyle='--', alpha=0.7)

        ax.set_xlabel('Epoch')
        ax.set_ylabel('Contrastive Loss')
        ax.set_title(f'Fold {fold_idx}  (best val={best_val:.4f} @ epoch {int(best_epoch)})')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    # Sembunyikan axes kosong
    for ax_idx in range(n_folds, n_rows * n_cols):
        row, col = divmod(ax_idx, n_cols)
        axes[row][col].set_visible(False)

    plt.suptitle('Training Loss Curves', fontsize=13, y=1.01)
    plt.tight_layout()

    if save_dir:
        save_path = Path(save_dir) / 'training_curves.png'
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved: {save_path}')

    plt.show()


plot_training_history(all_histories, save_dir=OUTPUT_DIR)

In [ ]:
# ── Baca log CSV dan tampilkan sebagai tabel ──────────────────────────────────
print('Log training tersimpan di:')
for fold_idx, _ in all_histories:
    log_path = OUTPUT_DIR / f'fold_{fold_idx}' / 'train_log.csv'
    if log_path.exists():
        df = pd.read_csv(log_path)
        best_row = df.loc[df['val_loss'].idxmin()]
        print(f'  Fold {fold_idx}: {log_path}')
        print(f'    → Best epoch: {int(best_row["epoch"])} '
              f'(phase {int(best_row["phase"])})  '
              f'val_loss={best_row["val_loss"]:.4f}  '
              f'train_loss={best_row["train_loss"]:.4f}')
    else:
        print(f'  Fold {fold_idx}: log tidak ditemukan')

print(f'\nCheckpoint disimpan di: {OUTPUT_DIR}')
print('Jalankan evaluate.ipynb untuk evaluasi lengkap.')